# Phase 1: Data Collection

Pulls historical NAV data for 6 mutual funds across 3 AMCs and 2 categories
using mftool, cleans it (datetime parsing, numeric conversion, sorting),
and saves to Data/raw/ as CSV.

Funds: HDFC/SBI/ICICI × Large Cap, HDFC/ICICI × Corporate Bond, HDFC Hybrid Equity

In [1]:
import pandas as pd
from mftool import Mftool

mf = Mftool()
search_results = mf.get_scheme_codes()

In [22]:
funds = {
    "hdfc_large_cap": 119018,
    "hdfc_corporate_bond": 118987,
    "hdfc_hybrid_equity": 119062,
    "sbi_large_cap" : 119598,
    "icici_large_cap" : 120586,
    "icici_corporate_bond" : 120692
}

In [23]:
for name, code in funds.items():
    print(name, code)

hdfc_large_cap 119018
hdfc_corporate_bond 118987
hdfc_hybrid_equity 119062
sbi_large_cap 119598
icici_large_cap 120586
icici_corporate_bond 120692


In [24]:
for name, code in funds.items():
    raw = mf.get_scheme_historical_nav(code)

    df = pd.DataFrame(raw["data"])
    df['date'] = pd.to_datetime(df['date'] , format = '%d-%m-%Y')
    df['nav'] = pd.to_numeric(df['nav'] , errors = 'coerce')
    df =df.sort_values('date').reset_index(drop =True)
    df.to_csv(f"../Data/raw/{name}.csv", index = False)
    print(f"Saved {name} : {df.shape[0]} rows, {df['date'].min().date()} to {df['date'].max().date()}")

Saved hdfc_large_cap : 3329 rows, 2013-01-01 to 2026-07-14
Saved hdfc_corporate_bond : 3267 rows, 2013-01-01 to 2026-07-14
Saved hdfc_hybrid_equity : 3329 rows, 2013-01-01 to 2026-07-14
Saved sbi_large_cap : 3338 rows, 2013-01-02 to 2026-07-15
Saved icici_large_cap : 3329 rows, 2013-01-02 to 2026-07-15
Saved icici_corporate_bond : 3266 rows, 2013-01-03 to 2026-07-15


In [25]:
import os
# print(os.getcwd())
# print(os.path.exists("../Data/raw"))
# os.makedirs("../Data/raw" , exist_ok = True)
# print(os.path.exists("../Data"))
print(os.listdir("../Data/raw"))

['hdfc_corporate_bond.csv', 'hdfc_hybrid_equity.csv', 'hdfc_large_cap.csv', 'icici_corporate_bond.csv', 'icici_large_cap.csv', 'sbi_large_cap.csv']


In [11]:
check_df = pd.read_csv('../Data/raw/hdfc_large_cap.csv')
print(check_df.head())
print(check_df.tail())
print(check_df.dtypes)

         date      nav
0  2013-01-01  228.943
1  2013-01-02  231.032
2  2013-01-03  231.435
3  2013-01-04  232.035
4  2013-01-07  231.066
            date       nav
3324  2026-07-08  1218.468
3325  2026-07-09  1227.566
3326  2026-07-10  1236.776
3327  2026-07-13  1237.037
3328  2026-07-14  1228.902
date        str
nav     float64
dtype: object


In [26]:
check_sbi = pd.read_csv("../Data/raw/sbi_large_cap.csv")
print(check_sbi.head())
print(check_sbi.tail())

         date    nav
0  2013-01-02  16.83
1  2013-01-03  16.86
2  2013-01-04  16.88
3  2013-01-07  16.80
4  2013-01-08  16.84
            date       nav
3333  2026-07-09  103.9998
3334  2026-07-10  104.9543
3335  2026-07-13  104.8913
3336  2026-07-14  104.1665
3337  2026-07-15  104.6294


In [21]:
search_results = mf.get_scheme_codes()
for code, scheme_name in search_results.items():
    # if 'sbi' in scheme_name.lower() and 'large' in scheme_name.lower():
    # if 'icici' in scheme_name.lower() and 'large' in scheme_name.lower():
    if 'icici' in scheme_name.lower() and 'corporate bond' in scheme_name.lower():
        print(code, scheme_name)

130947 ICICI Prudential Corporate Bond Fund - Bonus
111988 ICICI Prudential Corporate Bond Fund - Daily IDCW
120695 ICICI Prudential Corporate Bond Fund - Direct Plan - Daily IDCW
120696 ICICI Prudential Corporate Bond Fund - Direct Plan - Fortnightly IDCW
120692 ICICI Prudential Corporate Bond Fund - Direct Plan - Growth
131152 ICICI Prudential Corporate Bond Fund - Direct Plan - Half Yearly IDCW Option
120697 ICICI Prudential Corporate Bond Fund - Direct Plan - Monthly IDCW
120694 ICICI Prudential Corporate Bond Fund - Direct Plan - Quarterly IDCW
120693 ICICI Prudential Corporate Bond Fund - Direct Plan - Weekly IDCW
111990 ICICI Prudential Corporate Bond Fund - Fortnightly IDCW
111987 ICICI Prudential Corporate Bond Fund - Growth
131151 ICICI Prudential Corporate Bond Fund - Half Yearly IDCW Option
111991 ICICI Prudential Corporate Bond Fund - Monthly IDCW
111982 ICICI Prudential Corporate Bond Fund - Premium Daily Dividend
111985 ICICI Prudential Corporate Bond Fund - Premium Mont

In [3]:
all_schemes = mf.get_scheme_codes()
print(len(all_schemes))
list(all_schemes.items())[:5]

14228


[('Scheme Code', 'Scheme Name'),
 ('119551', 'Aditya Birla Sun Life Banking & PSU Debt Fund  - DIRECT - IDCW'),
 ('119552',
  'Aditya Birla Sun Life Banking & PSU Debt Fund  - DIRECT - MONTHLY IDCW'),
 ('119553',
  'Aditya Birla Sun Life Banking & PSU Debt Fund  - Direct - Quarterly IDCW'),
 ('108272', 'Aditya Birla Sun Life Banking & PSU Debt Fund  - REGULAR - IDCW')]

In [4]:
print(all_schemes.get('119081'))

HDFC Medium Term Debt Fund - Growth Option - Direct Plan


                                **DATA COLLECTION PHASE 2**NEW
NEW DATA OF 140 FUNDS

In [2]:
# !pip install pyarrow

scheme_df = pd.read_csv('../Data/external/mutual_fund_data.csv')
print(scheme_df.shape)
print(scheme_df.columns.tolist())
scheme_df.head()

nav_history_df = pd.read_parquet('../Data/external/mutual_fund_nav_history.parquet')
print(nav_history_df.shape)
print(nav_history_df.columns.tolist())
nav_history_df.head()

(16325, 16)
['Scheme_Code', 'Scheme_Name', 'AMC', 'Scheme_Type', 'Scheme_Category', 'Scheme_NAV_Name', 'Scheme_Min_Amt', 'NAV', 'Latest_NAV_Date', 'Average_AUM_Cr', 'AAUM_Quarter', 'ISIN_Div_Payout/Growth', 'ISIN_Div_Reinvestment', 'ISIN_Div_Payout/Growth/Div_Reinvestment', 'Launch_Date', 'Closure_Date']
(21898166, 2)
['Date', 'NAV']


,Date,NAV
Scheme_Code,,
100033,2006-04-03,116.61
100033,2013-08-05,143.27
100033,2007-04-12,120.13
100033,2006-06-06,94.70
100033,2007-10-16,165.61


In [8]:
import os
print(os.getcwd())
print(os.listdir('../Data'))

d:\python stuff\investment_advisor_project\Notebooks
['external', 'raw']


In [3]:
icici_check = nav_history_df.loc[120586].sort_values('Date')
print(icici_check.shape)
print(icici_check.head())
print(icici_check.tail())

(3244, 2)
                  Date    NAV
Scheme_Code                  
120586      2013-01-02  18.66
120586      2013-01-03  18.73
120586      2013-01-04  18.76
120586      2013-01-07  18.72
120586      2013-01-08  18.76
                  Date     NAV
Scheme_Code                   
120586      2026-07-03  120.55
120586      2026-07-07  121.15
120586      2026-07-08  118.77
120586      2026-07-09  119.21
120586      2026-07-10  120.45


This above data matched the one we had in the previous 6 funds which ensures trust and credibility of data.

In [5]:
original_icici = pd.read_csv('../Data/raw/icici_large_cap.csv', parse_dates = ['date'])

print(original_icici.head())

        date    nav
0 2013-01-02  18.66
1 2013-01-03  18.73
2 2013-01-04  18.76
3 2013-01-07  18.72
4 2013-01-08  18.76


In [ ]:
# nunique gives count of uniquw cats and unique give the unique elements
print(scheme_df['Scheme_Category'].nunique())
print(scheme_df['Scheme_Category'].value_counts())
clean_schemes = scheme_df[scheme_df['Scheme_NAV_Name'].str.contains('Direct', case = False, na= False) &
                    scheme_df['Scheme_NAV_Name'].str.contains('Growth', case = False, na =False)
                    ].copy()
print(clean_schemes.shape)
scheme_df = clean_schemes
print(scheme_df.shape)


65
Scheme_Category
Income                                              807
Other Scheme - Index Funds                          354
Equity Scheme - Sectoral/ Thematic                  253
Other Scheme - FoF Domestic                         142
Other Scheme - FoF Overseas                          53
                                                   ... 
Hybrid Schemes - Aggressive Hybrid Fund               1
Equity Schemes - Large & Mid Cap Fund                 1
Index Funds - Debt Funds                              1
Equity Schemes - Mid Cap Fund                         1
Income/Debt Oriented Schemes - Other Debt Scheme      1
Name: count, Length: 65, dtype: int64
(2681, 16)
(2681, 16)


In [15]:
category_map = {
    'large_cap': 'Large Cap',
    'mid_cap': 'Mid Cap',
    'small_cap': 'Small Cap',
    'corporate_bond': 'Corporate Bond',
    'short_duration': 'Short Duration',
    'hybrid': 'Hybrid',
    'index': 'Index'
}

for label, keyword in category_map.items():
    matches = scheme_df['Scheme_Category'].unique()
    matched_cat = [c for c in matches if keyword.lower() in c.lower()]
    print(f"{label} : ({keyword}):")
    for c in matched_cat:
        count = (scheme_df['Scheme_Category'] == c).sum() 
        print( f'{c} : {count}')
    print()



large_cap : (Large Cap):
Equity Scheme - Large Cap Fund : 36

mid_cap : (Mid Cap):
Equity Scheme - Large & Mid Cap Fund : 37
Equity Scheme - Mid Cap Fund : 34
Equity Schemes - Large & Mid Cap Fund : 1
Equity Schemes - Mid Cap Fund : 1

small_cap : (Small Cap):
Equity Scheme - Small Cap Fund : 36

corporate_bond : (Corporate Bond):
Debt Scheme - Corporate Bond Fund : 22
Income/Debt Oriented Schemes - Corporate Bond Fund : 1

short_duration : (Short Duration):
Debt Scheme - Ultra Short Duration Fund : 27
Debt Scheme - Short Duration Fund : 29

hybrid : (Hybrid):
Hybrid Scheme - Aggressive Hybrid Fund : 31
Hybrid Scheme - Conservative Hybrid Fund : 22
Hybrid Scheme - Arbitrage Fund : 38
Hybrid Scheme - Equity Savings : 29
Hybrid Scheme - Dynamic Asset Allocation or Balanced Advantage : 36
Hybrid Scheme - Multi Asset Allocation : 37
Hybrid Scheme - Balanced Hybrid Fund : 3
Hybrid Schemes - Balanced Advantage Fund/ Dynamic Asset Allocation : 1
Hybrid Schemes - Aggressive Hybrid Fund : 1
Hyb

In [ ]:
category_filters = {
    'large_cap': ['Equity Scheme - Large Cap Fund', 'Equity Schemes - Large Cap Fund'],
    'mid_cap': ['Equity Scheme - Mid Cap Fund', 'Equity Schemes - Mid Cap Fund'],
    'small_cap': ['Equity Scheme - Small Cap Fund', 'Equity Schemes - Small Cap Fund'],
    'corporate_bond': ['Debt Scheme - Corporate Bond Fund', 'Income/Debt Oriented Schemes - Corporate Bond Fund'],
    'short_duration': ['Debt Scheme - Short Duration Fund'],
    'hybrid': ['Hybrid Scheme - Aggressive Hybrid Fund', 'Hybrid Schemes - Aggressive Hybrid Fund'],
}

category_dfs = {}
for label, cats in category_filters.items():
    sub = scheme_df[scheme_df['Scheme_Category'].isin(cats)]
    category_dfs[label] = sub
    print(f"{label}: {len(sub)} funds")


index_candidates = scheme_df[scheme_df['Scheme_Category'].isin([
    'Other Scheme - Index Funds', 'Index Funds - Equity Funds'
])]

index_equity = index_candidates[
    index_candidates['Scheme_NAV_Name'].str.contains('nifty|sensex', case=False, regex=True, na=False)
]                                                                               #regex tell to use the '|' as or and not a normal string in the 1st section

print(f"index (Nifty/Sensex only): {len(index_equity)} funds")
category_dfs['index'] = index_equity

sampled_funds = {}
for label, df in category_dfs.items():
    n = min(20, len(df))
    sampled_funds[label] = df.sample(n=n, random_state=42) #takes random n rows
    print(f"Sampled {len(sampled_funds[label])} from {label}")

final_selection = pd.concat(sampled_funds.values(), ignore_index=True)
print(f"\nTotal funds selected: {len(final_selection)}")
final_selection[['Scheme_Code', 'Scheme_NAV_Name', 'Scheme_Category']].head(20)

large_cap: 36 funds
mid_cap: 35 funds
small_cap: 36 funds
corporate_bond: 23 funds
short_duration: 29 funds
hybrid: 32 funds
index (Nifty/Sensex only): 275 funds
Sampled 20 from large_cap
Sampled 20 from mid_cap
Sampled 20 from small_cap
Sampled 20 from corporate_bond
Sampled 20 from short_duration
Sampled 20 from hybrid
Sampled 20 from index

Total funds selected: 140


,Scheme_Code,Scheme_NAV_Name,Scheme_Category
0,154307,JioBlackRock Large Cap Fund - Direct Plan - Gr...,Equity Scheme - Large Cap Fund
1,119598,SBI Large Cap FUND-DIRECT PLAN -GROWTH,Equity Scheme - Large Cap Fund
2,148507,Sundaram Large Cap Fund (Formerly Known as Sun...,Equity Scheme - Large Cap Fund
3,150797,WhiteOak Capital Large Cap Fund Direct Plan Gr...,Equity Scheme - Large Cap Fund
4,120267,LIC MF Large Cap Fund-Direct Plan-Growth,Equity Scheme - Large Cap Fund
5,152354,Motilal Oswal Large Cap Direct Plan Growth,Equity Scheme - Large Cap Fund
6,120656,UTI Large Cap Fund - Direct Plan - Growth Option,Equity Scheme - Large Cap Fund
7,119528,Aditya Birla Sun Life Large Cap Fund - Growth ...,Equity Scheme - Large Cap Fund
8,119018,HDFC Large Cap Fund - Growth Option - Direct Plan,Equity Scheme - Large Cap Fund
9,120392,Invesco India Largecap Fund - Direct Plan - Gr...,Equity Scheme - Large Cap Fund


In [17]:
# Sanity check: is scheme_df actually filtered right now?
print(len(scheme_df))
print(scheme_df['Scheme_NAV_Name'].str.contains('regular', case=False, na=False).sum())
print(scheme_df['Scheme_NAV_Name'].str.contains('growth', case=False, na=False).sum())

2681
5
2681


In [ ]:
selected_codes = final_selection['Scheme_Code'].tolist()
print(len(selected_codes), len(set(selected_codes)))

nav_selected = nav_history_df.loc[nav_history_df.index.isin(selected_codes)].copy()
print(nav_selected.shape)
nav_selected= nav_selected.reset_index()
nav_selected = nav_selected.sort_values(['Scheme_Code', 'Date']).reset_index(drop = True)
nav_selected.head()
print(nav_selected.shape)

140 140
(286236, 2)
(286236, 3)


In [22]:
print(nav_history_df.columns.tolist())
print(nav_history_df.index)
print(nav_history_df.index.names)
print(nav_history_df.shape)

['Date', 'NAV']
Index([100033, 100033, 100033, 100033, 100033, 100033, 100033, 100033, 100033,
       100033,
       ...
       154428, 154431, 154431, 154431, 154431, 154431, 154462, 154462, 154462,
       154462],
      dtype='int64', name='Scheme_Code', length=21898166)
['Scheme_Code']
(21898166, 2)


In [25]:
nav_selected = nav_selected.rename(columns={ 'Date': 'date', 'NAV': 'nav' })

In [26]:
nav_selected.to_csv('../Data/external/nav_selected_140funds.csv', index = False)
final_selection.to_csv('../Data/external/final_selection_140funs.csv', index = False)